# FraudShield — Model Training & Evaluation

Training multiple classifiers on the Kaggle Credit Card Fraud dataset with SMOTE oversampling.

**Models**: Logistic Regression, Random Forest, XGBoost
**Key challenge**: Extreme class imbalance (0.17% fraud)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, auc)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load & Split

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]}  Test: {X_test.shape[0]}')
print(f'Train fraud ratio: {y_train.mean()*100:.4f}%')
print(f'Test fraud ratio:  {y_test.mean()*100:.4f}%')

## 2. Feature Scaling

V1–V28 are already PCA-scaled. Only Time and Amount need normalization.

In [ ]:
scaler = StandardScaler()
X_train['Time'] = scaler.fit_transform(X_train[['Time']])
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Time'] = scaler.transform(X_test[['Time']])
X_test['Amount'] = scaler.transform(X_test[['Amount']])

## 3. SMOTE Oversampling

Synthetic Minority Over-sampling Technique generates synthetic fraud examples to balance the classes.

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {X_train.shape[0]} ({y_train.mean()*100:.2f}% fraud)')
print(f'After SMOTE:  {X_train_res.shape[0]} ({y_train_res.mean()*100:.2f}% fraud)')

## 4. Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'),
    'XGBoost': XGBClassifier(n_estimators=100, 
                             scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
                             random_state=42, eval_metric='logloss'),
}

results = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model, 'pred': y_pred, 'proba': y_proba,
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }
    print(f'\n--- {name} ---')
    print(f'  Precision: {results[name]["precision"]:.4f}')
    print(f'  Recall:    {results[name]["recall"]:.4f}')
    print(f'  F1:        {results[name]["f1"]:.4f}')
    print(f'  ROC-AUC:   {results[name]["roc_auc"]:.4f}')

## 5. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(10, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {r["roc_auc"]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Credit Card Fraud Detection')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Key Takeaways

- **Logistic Regression** has high recall (catches most fraud) but terrible precision (too many false alarms). Not practical alone.
- **Random Forest** is the most balanced — good precision and recall. Best for production if false positives are costly.
- **XGBoost** achieves the highest ROC-AUC. With threshold tuning, it's the strongest model.
- SMOTE was essential — without it the models would predict 'legitimate' for every transaction.

The pipeline uses XGBoost as default (best overall discrimination), with the API serving all three for comparison.